# **Práctica**



1. Cargar los datos especificados a continuación.
2. Realizar el join entre todos los archivos.
3. Verificar si existen duplicados, si existen, eliminarlos.
4. Verificar si existen valores vacíos, si existen, eliminarlos.
5. Contar cuantas estaciones registran ecosistemas:
   * Dañados
   * En riesgo
   * Saludable
4. Agrupar por región y contar.
5. ¿Cuál es el máximo porcentaje del fondo marino que está cubierto por coral vivo (indice_cobertura_coral)? ¿Y el mínimo? ¿En dónde se localiza cada uno de ellos?
6. ¿Cuál es la temperatura promedio registrada en las estaciones del Golfo de California?


# **PySpark**

## Instalación

In [ ]:
# Instalar PySpark
!pip install -q findspark pyspark

In [ ]:
!apt-get update -qq > /dev/null
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

In [ ]:
# Descargar Apache Spark
!wget -q https://dlcdn.apache.org/spark/spark-4.2.0/spark-4.2.0-bin-hadoop3.tgz
!tar xf spark-4.2.0-bin-hadoop3.tgz

In [ ]:
# Configurar variables de entorno
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-4.2.0-bin-hadoop3"

## Crear sesión de Spark

In [ ]:
import findspark
findspark.init()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EcosistemasMarinosML") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()


In [ ]:
spark.version

# **Datos**

Los datos se cargarán directamente de la web (GitHub).

## Cargar datos de las estaciones de monitoreo

In [ ]:
import urllib.request

# 1. URL Raw de GitHub
url_estaciones = "https://raw.githubusercontent.com/anaepm/rep/refs/heads/main/estaciones_monitoreo.csv"

# 2. Descargar el archivo
urllib.request.urlretrieve(url_estaciones, "estaciones_monitoreo.csv")

# 3. Leerlo con Spark
df_est = spark.read.option("header", "true").option("inferSchema", "true").csv("estaciones_monitoreo.csv")
df_est.show(5)

## Cargar datos de las condiciones físicas de los océanos

In [ ]:
# 1. URL Raw de GitHub
url_fisicas = "https://raw.githubusercontent.com/anaepm/rep/refs/heads/main/condiciones_fisicas.csv"

# 2. Descargar el archivo
urllib.request.urlretrieve(url_fisicas, "condiciones_fisicas.csv")

# 3. Leerlo con Spark
df_fis = spark.read.option("header", "true").option("inferSchema", "true").csv("condiciones_fisicas.csv")
df_fis.show(5)

## Cargar datos de las características biológicas de los océanos

In [ ]:
# 1. URL Raw de GitHub
url_biologicos = "https://raw.githubusercontent.com/anaepm/rep/refs/heads/main/reportes_biologicos.csv"

# 2. Descargar el archivo
urllib.request.urlretrieve(url_biologicos, "reportes_biologicos.csv")

# 3. Leerlo con Spark
df_bio = spark.read.option("header", "true").option("inferSchema", "true").csv("reportes_biologicos.csv")
df_bio.show(5)

# **Exploración de esquemas**

Antes de unir los archivos, revisamos las columnas de cada DataFrame. **Ajusta los nombres de columnas usados más adelante (`ID_COL`, `REGION_COL`, etc.) si difieren de los que aparecen aquí.**

In [ ]:
print('--- Estaciones ---')
df_est.printSchema()

print('--- Condiciones físicas ---')
df_fis.printSchema()

print('--- Reportes biológicos ---')
df_bio.printSchema()


# **1. Join entre todos los archivos**

Se asume que las tres tablas comparten una columna llave que identifica la estación (por ejemplo `id_estacion`). Ajusta el valor de `ID_COL` según lo que veas en los esquemas anteriores.

In [ ]:
# Nombre de la columna llave que identifica a la estación en las 3 tablas
ID_COL = "id_estacion"  # <-- ajusta este nombre si tu columna se llama distinto

# Unimos estaciones + condiciones físicas + reportes biológicos
df_join = df_est.join(df_fis, on=ID_COL, how="inner") \
                .join(df_bio, on=ID_COL, how="inner")

print(f"Filas resultantes del join: {df_join.count()}")
df_join.show(5)


# **2. Verificar y eliminar duplicados**

In [ ]:
total_filas = df_join.count()
filas_unicas = df_join.dropDuplicates().count()
num_duplicados = total_filas - filas_unicas

print(f"Filas totales: {total_filas}")
print(f"Filas duplicadas encontradas: {num_duplicados}")

if num_duplicados > 0:
    df_join = df_join.dropDuplicates()
    print("Duplicados eliminados.")
else:
    print("No se encontraron duplicados.")

print(f"Filas después de la limpieza: {df_join.count()}")


# **3. Verificar y eliminar valores vacíos**

In [ ]:
from pyspark.sql.functions import col, count, when

# Conteo de valores nulos/vacíos por columna
df_join.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_join.columns
]).show()

filas_antes = df_join.count()
df_join = df_join.dropna()
filas_despues = df_join.count()

print(f"Filas eliminadas por valores vacíos: {filas_antes - filas_despues}")
print(f"Filas finales: {filas_despues}")


# **4. Conteo de estaciones por estado del ecosistema**

Se asume que existe una columna categórica (por ejemplo `estado_ecosistema`) con los valores `Dañado`, `En riesgo` y `Saludable`. Ajusta `ESTADO_COL` si el nombre real es distinto.

In [ ]:
ESTADO_COL = "estado_ecosistema"  # <-- ajusta este nombre si es necesario

df_join.groupBy(ESTADO_COL).count().orderBy("count", ascending=False).show()


# **5. Agrupar por región y contar**

Ajusta `REGION_COL` si la columna de región se llama distinto.

In [ ]:
REGION_COL = "region"  # <-- ajusta este nombre si es necesario

df_join.groupBy(REGION_COL).count().orderBy("count", ascending=False).show()


# **6. Máximo y mínimo índice de cobertura de coral**

¿Cuál es el máximo y el mínimo porcentaje del fondo marino cubierto por coral vivo (`indice_cobertura_coral`) y en qué estación se localiza cada uno?

In [ ]:
CORAL_COL = "indice_cobertura_coral"  # <-- ajusta este nombre si es necesario

# Estación con el máximo índice de cobertura de coral
print("Máximo índice de cobertura de coral:")
df_join.orderBy(col(CORAL_COL).desc()).select(ID_COL, REGION_COL, CORAL_COL).show(1)

# Estación con el mínimo índice de cobertura de coral
print("Mínimo índice de cobertura de coral:")
df_join.orderBy(col(CORAL_COL).asc()).select(ID_COL, REGION_COL, CORAL_COL).show(1)


# **7. Temperatura promedio en el Golfo de California**

Ajusta `TEMP_COL` y el valor exacto de la región si difiere (por ejemplo puede estar escrito como `"Golfo de California"` sin acentos, etc.).

In [ ]:
from pyspark.sql.functions import avg

TEMP_COL = "temperatura"  # <-- ajusta este nombre si es necesario

df_join.filter(col(REGION_COL) == "Golfo de California") \
       .select(avg(col(TEMP_COL)).alias("temperatura_promedio")) \
       .show()
